# DES Y3 source-clustering calibration (Gatti et al. 2307.13860)

Determines the per-tomographic-bin calibration factors `(corr_variance, A_corr, coeff_kurtosis)`
for the `"gatti"` source-clustering mode of `forward_model_cosmogrid`, as linear functions of the
source-clustering bias `b_sc`, and saves them to `data/desy3_sc_calibration_gatti.npy` in the format that
`msfm.utils.files.read_sc_calibration` expects.

The `"gatti"` shape noise is the rotate-in-place noise modulated per pixel by
`f = 1 / sqrt(1 + b_sc * delta)` (see `lensing.source_clustering_factor`), so the *uncalibrated*
per-pixel noise variance is `x = f**2 * var_ref`, where `var_ref` is the cosmology-independent
reference shape-noise variance (`lensing.shape_noise_variance_map`). The calibration multiplies the
noise ellipticity by `mod = 1/sqrt(A_corr * corr_variance) * sqrt(1 + coeff_kurtosis * var_ref)` so
that the calibrated variance matches `var_ref` in both mean and second moment.

We deliberately use `var_ref` (the reference variance) in the kurtosis term in **both** the
calibration and the pipeline, resolving the x-vs-y mismatch in the Gatti reference notebook (which
is the read-only DES Y6 reference under `Gatti/`).

This calibration is averaged over several fiducial CosmoGrid realizations (section 2) for a more
robust fit: `var_ref` is realization-independent (built from the real catalog), so only the
simulation density contrast `delta` varies, and section 3 pools all realizations. Set the `perms`
list in section 2 to a single directory to reproduce the original single-realization fit.

In [ ]:
import os
import glob
import h5py
import numpy as np
from scipy.optimize import minimize

from msfm.utils import files, lensing, filenames, imports

hp = imports.import_healpy()

## 1. Config, footprint and the reference shape-noise variance

`var_ref` is built from the real catalog (the noise file) and is independent of cosmology and `b_sc`.

In [4]:
conf = files.load_config("/users/athomsen/dlss/repos/multiprobe-simulation-forward-model/configs/v16/gatti_sc.yaml")

n_side = conf["analysis"]["n_side"]
n_pix = hp.nside2npix(n_side)
metacal_bins = conf["survey"]["metacal"]["z_bins"]
n_z = len(metacal_bins)

# base survey patch per tomographic bin (i_patch = 0, where cutout_patch_pix == patch_pix, matching
# the i_patch = 0 path of forward_model_cosmogrid)
_, patches_pix_dict, _, _ = files.load_pixel_file(conf)

# reference (no source-clustering) shape-noise variance per pixel, per metacal bin
tomo_gamma_cat = files.load_noise_file(conf)
var_ref = np.stack(
    [
        lensing.shape_noise_variance_map(np.abs(c[:, 0] + 1j * c[:, 1]), c[:, 2], c[:, 3], n_pix)
        for c in tomo_gamma_cat
    ],
    axis=-1,
)

26-06-29 15:32:25     files.py INF   Loaded the noise file 


## 2. Simulation source-bin density contrast (averaged over realizations)

Load a set of fiducial full-sky CosmoGrid maps (the same `.h5` that `forward_model_cosmogrid`
reads). `delta` is computed exactly as in the `"gatti"` branch of `forward_model_cosmogrid`, one
per realization. The fit in section 3 pools all realizations, which averages the variance/kurtosis
moments over the ensemble for a more robust calibration. `var_ref` (section 1) is
realization-independent, so it is not re-loaded here. Set `perms` to a single directory to
reproduce the original single-realization fit.

In [ ]:
# fiducial CosmoGrid realizations to average over (set to a single perm to reproduce the old fit)
fiducial_dir = "/users/athomsen/scratch/deep_lss/data/projected/fiducial/cosmo_fiducial"
perms = sorted(glob.glob(os.path.join(fiducial_dir, "perm_*")))[:5]
assert len(perms) > 0, f"No perm_* directories found under {fiducial_dir}"
print(f"Averaging over {len(perms)} realizations: {[os.path.basename(p) for p in perms]}")

deltas = []
for map_dir in perms:
    map_file = filenames.get_filename_full_maps(map_dir, with_bary=conf["analysis"]["modelling"]["baryonified"])
    with h5py.File(map_file, "r") as f:
        dg = np.stack([hp.ud_grade(f[f"map/dg/{z_bin}"], n_side) for z_bin in metacal_bins], axis=-1)
    deltas.append((dg - np.mean(dg, axis=0)) / np.mean(dg, axis=0))

# (n_realizations, n_pix, n_z)
deltas = np.stack(deltas, axis=0)

## 3. Fit `(corr_variance, A_corr, coeff_kurtosis)` for a grid of `b_sc`

For each `b_sc` and bin we match the calibrated `gatti` variance to the reference variance in mean
(variance ratio) and second moment (kurtosis ratio), as in the Gatti reference. The per-pixel
uncalibrated variance `x = f**2 * var_ref` is pooled across all realizations (`var_ref` is shared),
so the fit matches the ensemble-averaged moments rather than those of a single realization.

In [ ]:
b_grid = np.linspace(0.5, 1.5, 11)
n_real = deltas.shape[0]

calib = {name: {b: np.zeros(n_z) for b in b_grid} for name in ["corr_variance", "A_corr", "coeff_kurtosis"]}

for b in b_grid:
    for i_z in range(n_z):
        patch_pix = patches_pix_dict["metacal"][i_z][0]
        vref = var_ref[patch_pix, i_z]

        # footprint pixels that actually contain galaxies (realization-independent)
        mask = vref > 0
        vref = vref[mask]

        # pool the uncalibrated gatti variance over all realizations; var_ref is shared, so the
        # pooled moments equal the ensemble average of the per-realization moments
        x = np.concatenate(
            [lensing.source_clustering_factor(delta[patch_pix, i_z][mask], b) ** 2 * vref for delta in deltas]
        )
        vref_pooled = np.tile(vref, n_real)

        corr_var = np.mean(x) / np.mean(vref_pooled)
        mean_y, mean_y2 = np.mean(vref_pooled), np.mean(vref_pooled**2)

        def objective(params, x=x, vref=vref_pooled, corr_var=corr_var, mean_y=mean_y, mean_y2=mean_y2):
            A, coeff_kurtosis = params
            mod2 = 1.0 / (A * corr_var) * (1.0 + coeff_kurtosis * vref)
            xc = x * mod2
            return (np.mean(xc) / mean_y - 1.0) ** 2 + (np.mean(xc**2) / mean_y2 - 1.0) ** 2

        res = minimize(objective, [1.0, 0.0], bounds=[(0.9, 1.1), (-1.0, 1.0)])
        A_opt, coeff_kurtosis_opt = res.x

        calib["corr_variance"][b][i_z] = corr_var
        calib["A_corr"][b][i_z] = A_opt
        calib["coeff_kurtosis"][b][i_z] = coeff_kurtosis_opt

## 4. Linear fit in `b_sc` and save

`read_sc_calibration` evaluates `slope * b_sc + intercept` per bin per quantity.

In [7]:
fits = {}
for name in ["corr_variance", "A_corr", "coeff_kurtosis"]:
    slope = np.zeros(n_z)
    intercept = np.zeros(n_z)
    for i_z in range(n_z):
        y_vals = np.array([calib[name][b][i_z] for b in b_grid])
        slope[i_z], intercept[i_z] = np.polyfit(b_grid, y_vals, 1)
    fits[name] = {"slope": slope, "intercept": intercept}

file_dir = os.path.dirname(files.__file__)
repo_dir = os.path.abspath(os.path.join(file_dir, "../.."))
out_file = os.path.join(repo_dir, conf["files"]["sc_calibration"])
np.save(out_file, fits)
print(f"Saved source-clustering calibration to {out_file}")

# sanity check: read it back the way the pipeline does
files.read_sc_calibration(conf, b_sc=np.ones(n_z))

Saved source-clustering calibration to /users/athomsen/dlss/repos/multiprobe-simulation-forward-model/data/sc_calibration_desy3.npy
26-06-29 15:32:37     files.py INF   Loaded source-clustering calibration from /users/athomsen/dlss/repos/multiprobe-simulation-forward-model/data/sc_calibration_desy3.npy 


[(np.float64(1.059046038267978),
  np.float64(1.0015874500475008),
  np.float64(-0.6609923473918324)),
 (np.float64(1.0310197290892305),
  np.float64(1.0094975307427938),
  np.float64(-0.6363912748464193)),
 (np.float64(1.0208312654482765),
  np.float64(1.0046430086498055),
  np.float64(-0.09095464310857454)),
 (np.float64(1.013421269779583),
  np.float64(1.0044535175068574),
  np.float64(-2.657743917671334e-05))]

## 5. (Optional) end-to-end validation

With the calibration in place, generate noise-only maps for `source_clustering: gatti` via
`observation.forward_model_cosmogrid(map_dir, conf, noisy=True, noise_only=True, with_clustering=False,
tomo_bg_metacal=[...])` and confirm the per-pixel noise variance over the footprint matches the
`rotate` reference in mean and second moment. With `tomo_bg_metacal=[0, 0, 0, 0]` the `gatti` noise
must be identical to `rotate` (a no-op modulation).